In [108]:
#load The data
import pandas as pd

In [109]:
df = pd.read_csv("IMDB.csv")

In [110]:
df.shape

(50000, 2)

In [111]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [112]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

## Text Preprocessing

### 1.Converting to lowercase value

In [113]:
df['review'] = df["review"].str.lower()

### 2.Remove URLs

In [114]:
# import re

# sample_tex = "abc the word abc" # abc ==> xyz

# new_tex = re.sub("abc","xyz",sample_tex)
# new_tex

In [115]:
import re

def remove_urls(text):
    return re.sub(r"http\S+", "", str(text))

df["review"] = df["review"].apply(remove_urls)

### 3.Removing Punchuation

In [116]:
def remove_Panctuatoion(text):
    return re.sub(r"[^A-Za-z0-9\s]", "", str(text)) # A-Z , a-z, 1-0, 

df["review"] = df["review"].apply(remove_Panctuatoion)

### 4.Reomove HTML Tags

In [117]:
def remove_html(text):
    return re.sub(r"<.*?>", "", str(text)) # A-Z , a-z, 1-0, 

df["review"] = df["review"].apply(remove_html)

### 5.Removing the stopwords


In [120]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [121]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [122]:
# sampleText = "I love coding python"
# token = word_tokenize(sampleText)

In [123]:
def remove_stopwords(text):
    token = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in token:
        if word in stop_words:
            text.replace(word, "")

    return text


# Apply thi funchion
df["review"] = df["review"].apply(remove_stopwords)

In [124]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 6. Steming

In [125]:
# running ==> run
# played == > paly
# algo ===> (porterSteming)

In [126]:
from nltk.stem import PorterStemmer

In [129]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

# Apply thi funchion
df["review"] = df["review"].apply(stemming)

In [131]:
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,positive
1,a wonder littl product br br the film techniqu...,positive
2,i thought thi wa a wonder way to spend time on...,positive
3,basic there a famili where a littl boy jake th...,negative
4,petter mattei love in the time of money is a v...,positive


### 7.Encoding


In [133]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [134]:
y = df["sentiment"]
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8.Vectorization

In [140]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

## DataSet & DataLoaders

In [145]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [149]:
X_train.shape

(39665, 5000)

In [151]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4699361 stored elements and shape (39665, 5000)>

In [170]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [171]:
# X_train = X_train.toarray()
# X_test = X_train.toarray()

In [172]:
train_set = TensorDataset(
    # Directly creat tensor
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)
test_set = TensorDataset(
    # Directly creat tensor
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [173]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [181]:
import torch.nn as nn
import torch.optim as optim

In [196]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [199]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Train Our RNN

In [201]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1)# add singlton direction
        outputs = model(Xb) # {batch_size = 1}

        outputs = torch.sigmoid(outputs.squeeze()) # {batch_size, }

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # BP
        optimizer.step() # weights update

    print(f"epoch {epoch +1} / {epochs} & loss : {loss.item()} ")

epoch 1 / 10 & loss : 0.4356191158294678 
epoch 2 / 10 & loss : 0.08657436072826385 
epoch 3 / 10 & loss : 0.2130829393863678 
epoch 4 / 10 & loss : 0.21558336913585663 
epoch 5 / 10 & loss : 0.24757423996925354 
epoch 6 / 10 & loss : 0.13158205151557922 
epoch 7 / 10 & loss : 0.29062002897262573 
epoch 8 / 10 & loss : 0.15092410147190094 
epoch 9 / 10 & loss : 0.19557420909404755 
epoch 10 / 10 & loss : 0.0786108523607254 


# Evaluate

In [205]:
model.eval()

with torch.no_grad():
    correct_val = 0
    total_val = 0

    for Xb, yb in test_loader:

        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)

        # Apply sigmoid first, then threshold
        predicted = (torch.sigmoid(outputs).squeeze(-1) > 0.5).float()

        yb = yb.float().view(-1)

        total_val += yb.size(0)
        correct_val += (predicted == yb).sum().item()

    accuracy = correct_val / total_val * 100

    print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 87.68%
